In [1]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, classification_report, confusion_matrix, precision_score
import numpy as np
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import warnings
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.base import clone

In [2]:
df = pd.read_csv('foods_health_scores_allergens.csv')
df

,product_name,brands,categories,ingredients,nutriscore_grade,nova_group,ecoscore_grade,allergens,energy_kcal,fat_100g,...,proteins_100g,salt_100g,sodium_100g,contains_gluten,contains_dairy,contains_nuts,contains_soy,contains_eggs,contains_fish,food_type
0,Sidi Ali,سيدي علي,"en:beverages-and-beverages-preparations, en:be...",OBD1 999 999 1112606 266963207 mb,A,NaN,NOT-APPLICABLE,NaN,0.0,0.0,...,0.0,0.000000,0.000000,False,False,False,False,False,False,Branded/Packaged
1,Perly,Perly,"en:dairies, en:fermented-foods, en:fermented-m...","milk cream, cream, sugar, banana, bacteria",UNKNOWN,3.0,B,"en:banana, en:milk",97.0,3.0,...,8.0,NaN,NaN,False,True,False,False,False,False,Branded/Packaged
2,Sidi Ali,Sidi Ali,"en:beverages-and-beverages-preparations, en:be...","Sodium, Calcium, Magnésium, Potassium, Bicarbo...",A,1.0,NOT-APPLICABLE,NaN,NaN,NaN,...,NaN,0.065000,0.026000,False,False,False,False,False,False,Branded/Packaged
3,Eau minérale naturelle,sidi ali,"en:beverages-and-beverages-preparations, en:be...",100% mineral water,A,1.0,NOT-APPLICABLE,NaN,NaN,NaN,...,NaN,0.065000,0.026000,False,False,False,False,False,False,Branded/Packaged
4,اكوافينا,AQUAFINA,"en:beverages-and-beverages-preparations, en:be...",ouverture et avant le : Voir bouteille. après ...,A,NaN,NOT-APPLICABLE,NaN,0.0,0.0,...,0.0,0.000508,0.000203,False,False,False,False,False,False,Branded/Packaged
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4992,Crème fraîche gastronomique,Président,"en:dairies, en:fermented-foods, en:fermented-m...","_Crème_ (origine France), _ferments lactiques_",D,3.0,A,en:milk,291.0,30.0,...,2.3,0.070000,0.028000,False,True,False,False,False,False,Branded/Packaged
4993,Noix de cajou grillées sans sel,Maître Prunille,"en:plant-based-foods-and-beverages, en:plant-b...",Noix de cajou,B,1.0,E,en:nuts,621.0,47.0,...,21.0,0.020000,0.008000,False,False,True,False,False,False,Branded/Packaged
4994,Cacao puro en polvo desgrasado,Valor,"en:cocoa-and-its-products, en:cocoa-and-chocol...","Cacao desgrasado en polvo, correctores de acid...",C,1.0,F,NaN,375.0,16.0,...,26.0,0.030000,0.012000,False,False,False,False,False,False,Branded/Packaged
4995,Sablés Myrtilles Germes de riz,Gerblé,"en:snacks, en:sweet-snacks, en:biscuits-and-cakes","Farine de blé 58,2%, huile de colza, sucre de ...",A,4.0,C,"en:gluten, en:milk",54.0,2.0,...,0.9,0.050000,0.020000,True,True,False,False,False,False,Branded/Packaged


In [3]:
df["food_type"].value_counts()

food_type
Branded/Packaged    4997
Name: count, dtype: int64

In [4]:
df = df.drop(['product_name', 'brands', 'categories', 'ingredients', 'allergens', 'food_type'], axis = 1)

In [5]:
cols = [
    'contains_gluten',
    'contains_dairy',
    'contains_nuts',
    'contains_soy',
    'contains_eggs',
    'contains_fish'
]

df['Y'] = df[cols].any(axis=1)

In [6]:
df['Y'] = df['Y'].astype(int)

In [7]:
df["Y"]

0       0
1       1
2       0
3       0
4       0
       ..
4992    1
4993    1
4994    0
4995    1
4996    1
Name: Y, Length: 4997, dtype: int64

In [8]:
df = df.drop(cols, axis = 1)

In [9]:
df

,nutriscore_grade,nova_group,ecoscore_grade,energy_kcal,fat_100g,saturated_fat_100g,carbs_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,sodium_100g,Y
0,A,NaN,NOT-APPLICABLE,0.0,0.0,0.0,4.2,1.4,0.0,0.0,0.000000,0.000000,0
1,UNKNOWN,3.0,B,97.0,3.0,NaN,9.4,NaN,NaN,8.0,NaN,NaN,1
2,A,1.0,NOT-APPLICABLE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.065000,0.026000,0
3,A,1.0,NOT-APPLICABLE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.065000,0.026000,0
4,A,NaN,NOT-APPLICABLE,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000508,0.000203,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4992,D,3.0,A,291.0,30.0,22.0,2.9,2.9,NaN,2.3,0.070000,0.028000,1
4993,B,1.0,E,621.0,47.0,8.7,25.0,6.2,4.5,21.0,0.020000,0.008000,1
4994,C,1.0,F,375.0,16.0,10.0,16.0,0.7,32.0,26.0,0.030000,0.012000,0
4995,A,4.0,C,54.0,2.0,0.2,7.9,1.9,0.5,0.9,0.050000,0.020000,1


In [10]:
df.columns

Index(['nutriscore_grade', 'nova_group', 'ecoscore_grade', 'energy_kcal',
       'fat_100g', 'saturated_fat_100g', 'carbs_100g', 'sugars_100g',
       'fiber_100g', 'proteins_100g', 'salt_100g', 'sodium_100g', 'Y'],
      dtype='str')

In [11]:
y = df["Y"]
X = df.drop("Y", axis = 1)

In [12]:
y.value_counts(normalize = True)

Y
1    0.621573
0    0.378427
Name: proportion, dtype: float64

In [13]:
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y,
    test_size=0.2,        
    random_state=5831,      
    stratify=y           
)

In [14]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.25,        
    random_state=5831,      
    stratify=y           
)

In [15]:
baseline_recall = (y_test == 1).mean()
print("Baseline recall:", baseline_recall)

Baseline recall: 0.622


In [16]:
precision_threshold = 0.7

In [17]:
def adjust_threshold(pipe, X_train, y_train, X_val, y_val, X_test, y_test, X_trainval, y_trainval):
    # fit model initially on train only
    threshold_model = clone(pipe)
    threshold_model.fit(X_train, y_train)

    # get probabilities for validation set
    y_val_prob = threshold_model.predict_proba(X_val)[:, 1]

    # search over thresholds for validation set
    thresholds = np.arange(0.05, 0.96, 0.05)
    recalls = {}

    for t in thresholds:
        y_val_pred = (y_val_prob >= t).astype(int)
        precision = precision_score(y_val, y_val_pred, pos_label=1, zero_division=0)
        recall = recall_score(y_val, y_val_pred, pos_label=1, zero_division=0)

        if precision >= precision_threshold:
            recalls[t] = recall

    if len(recalls) == 0:
        raise ValueError(
            f"No threshold met the precision requirement of {precision_threshold}."
        )

    best_threshold = max(recalls, key=recalls.get)

    print(f"Threshold corresponding to max validation recall is {best_threshold}")
    print(f"Max validation recall is {recalls[best_threshold]}")

    final_model = clone(pipe)
    final_model.fit(X_trainval, y_trainval)

    # evaluate on test set
    y_test_prob = final_model.predict_proba(X_test)[:, 1]
    y_test_pred = (y_test_prob >= best_threshold).astype(int)

    recall = recall_score(y_test, y_test_pred, pos_label=1, zero_division=0)
    precision = precision_score(y_test, y_test_pred, pos_label=1, zero_division=0)

    print("Test Recall:", recall)
    print("Test Precision:", precision)

    cm = confusion_matrix(y_test, y_test_pred)

    cm_df = pd.DataFrame(
        cm,
        index=["Actual No Allergen", "Actual Has Allergens"],
        columns=["Pred No Allergen", "Pred Has Allergens"]
    )

    print(cm_df)

    return best_threshold, recall, precision, cm_df, final_model

In [18]:
results_dict = dict()

In [19]:
# define numeric and categorical columns
num_cols = ['energy_kcal','fat_100g', 'saturated_fat_100g', 'carbs_100g', 'sugars_100g','fiber_100g', 'proteins_100g', 'salt_100g', 'sodium_100g']     
cat_cols = ['nutriscore_grade', 'nova_group', 'ecoscore_grade']  

In [20]:
# define numeric pipeline
num_pipeline = Pipeline([
    ("imputer", KNNImputer(n_neighbors=5)),
    ("scaler", StandardScaler())
])

# define categorical pipeline
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# combine two pipelines
preprocess = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

# full pipeline for LDA
pipe = Pipeline([
    ("preprocess", preprocess),
    ("lda", LinearDiscriminantAnalysis())
])

In [21]:
threshold_lda, recall_lda, precision_lda, cm_df_lda, final_model_lda = adjust_threshold(pipe, X_train, y_train, X_val, y_val, X_test, y_test, X_trainval, y_trainval)

Threshold corresponding to max validation recall is 0.4
Max validation recall is 0.9536679536679536
Test Recall: 0.9565916398713826
Test Precision: 0.7066508313539193
                      Pred No Allergen  Pred Has Allergens
Actual No Allergen                 131                 247
Actual Has Allergens                27                 595


In [22]:
results_dict["LDA"] = {"recall":recall_lda, "precision": precision_lda, "threshold": threshold_lda}

In [23]:
result = permutation_importance(
    final_model_lda, X_test, y_test,
    n_repeats=10,
    random_state=5831,
    n_jobs=-1
)

importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance": result.importances_mean
}).sort_values(by="importance", ascending=False)

print(importance.head(10))

               feature  importance
0     nutriscore_grade      0.0633
2       ecoscore_grade      0.0580
1           nova_group      0.0323
3          energy_kcal      0.0314
9        proteins_100g      0.0276
4             fat_100g      0.0185
5   saturated_fat_100g      0.0134
7          sugars_100g      0.0084
11         sodium_100g      0.0034
8           fiber_100g      0.0013


In [24]:
num_pipeline = Pipeline([
    ("imputer", KNNImputer(n_neighbors=5)),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocess = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

pipe = Pipeline([
    ("preprocess", preprocess),
    ("qda", QuadraticDiscriminantAnalysis(reg_param=0.1))
])

In [25]:
threshold_qda, recall_qda, precision_qda, cm_df_qda, final_model_qda = adjust_threshold(pipe, X_train, y_train, X_val, y_val, X_test, y_test, X_trainval, y_trainval)

Threshold corresponding to max validation recall is 0.6000000000000001
Max validation recall is 0.9703989703989704
Test Recall: 0.9710610932475884
Test Precision: 0.7015098722415796
                      Pred No Allergen  Pred Has Allergens
Actual No Allergen                 121                 257
Actual Has Allergens                18                 604


In [26]:
results_dict["QDA"] = {"recall":recall_qda, "precision": precision_qda, "threshold": threshold_qda}

In [27]:
result = permutation_importance(
    final_model_qda, X_test, y_test,
    n_repeats=10,
    random_state=5831,
    n_jobs=-1
)

importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance": result.importances_mean
}).sort_values(by="importance", ascending=False)

print(importance.head(10))

               feature  importance
2       ecoscore_grade      0.0465
3          energy_kcal      0.0141
0     nutriscore_grade      0.0121
4             fat_100g      0.0091
5   saturated_fat_100g      0.0053
1           nova_group      0.0025
11         sodium_100g      0.0022
8           fiber_100g      0.0018
9        proteins_100g      0.0016
7          sugars_100g      0.0013


# Regularized LDA

In [28]:
num_pipeline = Pipeline([
    ("imputer", KNNImputer(n_neighbors=5)),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False))
])

preprocess = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

pipe = Pipeline([
    ("preprocess", preprocess),
    ("lda", LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto"))
])

In [29]:
threshold_reglda, recall_reglda, precision_reglda, cm_df_reglda, final_model_reglda = adjust_threshold(pipe, X_train, y_train, X_val, y_val, X_test, y_test, X_trainval, y_trainval)

Threshold corresponding to max validation recall is 0.5
Max validation recall is 0.9292149292149292
Test Recall: 0.932475884244373
Test Precision: 0.7169344870210136
                      Pred No Allergen  Pred Has Allergens
Actual No Allergen                 149                 229
Actual Has Allergens                42                 580


In [30]:
results_dict["Reg LDA"] = {"recall":recall_reglda, "precision": precision_reglda, "threshold": threshold_reglda}

In [31]:
result = permutation_importance(
    final_model_reglda, X_test, y_test,
    n_repeats=10,
    random_state=5831,
    n_jobs=-1
)

importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance": result.importances_mean
}).sort_values(by="importance", ascending=False)

print(importance.head(10))

               feature  importance
0     nutriscore_grade      0.0640
2       ecoscore_grade      0.0495
1           nova_group      0.0215
9        proteins_100g      0.0206
3          energy_kcal      0.0119
4             fat_100g      0.0046
7          sugars_100g      0.0035
10           salt_100g      0.0024
5   saturated_fat_100g      0.0020
11         sodium_100g      0.0019


In [32]:
num_pipeline = Pipeline([
    ("imputer", KNNImputer(n_neighbors=5))
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

pipe = Pipeline([
    ("preprocess", preprocess),
    ("xgb", XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=5831,
        eval_metric="logloss"
    ))
])

In [33]:
threshold_xgb, recall_xgb, precision_xgb, cm_df_xgb, final_model_xgb = adjust_threshold(pipe, X_train, y_train, X_val, y_val, X_test, y_test, X_trainval, y_trainval)

Threshold corresponding to max validation recall is 0.1
Max validation recall is 0.9884169884169884
Test Recall: 0.9903536977491961
Test Precision: 0.7359617682198327
                      Pred No Allergen  Pred Has Allergens
Actual No Allergen                 157                 221
Actual Has Allergens                 6                 616


In [34]:
results_dict["XGBoost"] = {"recall":recall_xgb, "precision": precision_xgb, "threshold": threshold_xgb}

In [35]:
result = permutation_importance(
    final_model_xgb, X_test, y_test,
    n_repeats=10,
    random_state=5831,
    n_jobs=-1
)

importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance": result.importances_mean
}).sort_values(by="importance", ascending=False)

print(importance.head(10))

               feature  importance
9        proteins_100g      0.1446
4             fat_100g      0.0482
7          sugars_100g      0.0473
3          energy_kcal      0.0408
6           carbs_100g      0.0369
10           salt_100g      0.0358
2       ecoscore_grade      0.0315
5   saturated_fat_100g      0.0315
8           fiber_100g      0.0290
11         sodium_100g      0.0246


In [36]:
num_pipeline = Pipeline([
    ("imputer", KNNImputer(n_neighbors=5))
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

pipe = Pipeline([
    ("preprocess", preprocess),
    ("lgbm", LGBMClassifier(
        n_estimators=200,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=5831
    ))
])

In [37]:
threshold_lgbm, recall_lgbm, precision_lgbm, cm_df_lgbm, final_model_lgbm = adjust_threshold(pipe, X_train, y_train, X_val, y_val, X_test, y_test, X_trainval, y_trainval)

[LightGBM] [Info] Number of positive: 2329, number of negative: 1418
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000367 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2290
[LightGBM] [Info] Number of data points in the train set: 3747, number of used features: 29
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.621564 -> initscore=0.496192
[LightGBM] [Info] Start training from score 0.496192


/opt/anaconda3/envs/CS5831FinalClassification/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Threshold corresponding to max validation recall is 0.05
Max validation recall is 0.990990990990991
[LightGBM] [Info] Number of positive: 2484, number of negative: 1513
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000199 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2304
[LightGBM] [Info] Number of data points in the train set: 3997, number of used features: 29
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.621466 -> initscore=0.495776
[LightGBM] [Info] Start training from score 0.495776
Test Recall: 0.9951768488745981
Test Precision: 0.70662100456621
                      Pred No Allergen  Pred Has Allergens
Actual No Allergen                 121                 257
Actual Has Allergens                 3                 619


/opt/anaconda3/envs/CS5831FinalClassification/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [38]:
results_dict["Lightgbm"] = {"recall":recall_lgbm, "precision": precision_lgbm, "threshold": threshold_lgbm}

In [39]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")

    result = permutation_importance(
    final_model_lgbm, X_test, y_test,
    n_repeats=10,
    random_state=5831,
    n_jobs=-1
    )

    importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance": result.importances_mean
    }).sort_values(by="importance", ascending=False)

    print(importance.head(10)) 

               feature  importance
9        proteins_100g      0.1626
4             fat_100g      0.0579
7          sugars_100g      0.0494
3          energy_kcal      0.0476
6           carbs_100g      0.0370
5   saturated_fat_100g      0.0367
10           salt_100g      0.0356
2       ecoscore_grade      0.0326
8           fiber_100g      0.0313
11         sodium_100g      0.0269


In [40]:
results = []

# transform and fit on training
X_train_trans = preprocess.fit_transform(X_train).astype(np.float32)
X_val_trans   = preprocess.transform(X_val).astype(np.float32)

y_train_np = y_train.to_numpy()
y_val_np   = y_val.to_numpy()

thresholds = np.arange(0.05, 0.96, 0.05)

# grid search
for n_d in [16, 32, 64]:
    for n_steps in [5, 7]:
        for lr in [0.05]:

            model = TabNetClassifier(
                seed=5831,
                n_d=n_d,
                n_a=n_d,
                n_steps=n_steps,
                gamma=1.5,
                lambda_sparse=1e-4,
                optimizer_params=dict(lr=lr),
                mask_type="entmax",
                verbose=0
            )

            model.fit(
                X_train_trans,
                y_train_np,
                eval_set=[(X_val_trans, y_val_np)],
                eval_name=["valid"],
                eval_metric=["auc"],
                max_epochs=200,
                patience=20,
                batch_size=256,
                virtual_batch_size=64,
                num_workers=0,
                drop_last=False
            )

            # find probabilities
            y_val_prob = model.predict_proba(X_val_trans)[:, 1]

            best_recall = -1
            best_t = None
            best_precision = None

            # threshold search
            for t in thresholds:
                y_val_pred_t = (y_val_prob >= t).astype(int)

                precision = precision_score(y_val, y_val_pred_t, pos_label=1, zero_division=0)
                recall = recall_score(y_val, y_val_pred_t, pos_label=1, zero_division=0)

                # must meet minimum precision threshold
                if precision >= precision_threshold:
                    if recall > best_recall:
                        best_recall = recall
                        best_t = t
                        best_precision = precision

            results.append({
                "n_d": n_d,
                "n_steps": n_steps,
                "lr": lr,
                "best_threshold": best_t,
                "valid_pos_recall": best_recall,
                "precision": best_precision
            })


Early stopping occurred at epoch 25 with best_epoch = 5 and best_valid_auc = 0.76852


/opt/anaconda3/envs/CS5831FinalClassification/lib/python3.12/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 89 with best_epoch = 69 and best_valid_auc = 0.83565


/opt/anaconda3/envs/CS5831FinalClassification/lib/python3.12/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 43 with best_epoch = 23 and best_valid_auc = 0.82294


...

/opt/anaconda3/envs/CS5831FinalClassification/lib/python3.12/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 32 with best_epoch = 12 and best_valid_auc = 0.80167


/opt/anaconda3/envs/CS5831FinalClassification/lib/python3.12/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 79 with best_epoch = 59 and best_valid_auc = 0.82627


/opt/anaconda3/envs/CS5831FinalClassification/lib/python3.12/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


In [41]:
results_df = pd.DataFrame(results).sort_values("valid_pos_recall", ascending=False)
print(results_df.head(10))

   n_d  n_steps    lr  best_threshold  valid_pos_recall  precision
2   32        5  0.05            0.50          0.980695   0.706209
1   16        7  0.05            0.20          0.979408   0.700737
4   64        5  0.05            0.55          0.974260   0.700926
3   32        7  0.05            0.25          0.972973   0.714556
5   64        7  0.05            0.35          0.970399   0.711321
0   16        5  0.05            0.70          0.797941   0.766378


In [42]:
best_row = results_df.iloc[0]

best_n_d = int(best_row["n_d"])
best_n_steps = int(best_row["n_steps"])
best_lr = float(best_row["lr"])
best_threshold = float(best_row["best_threshold"])

X_trainval_trans = preprocess.fit_transform(X_trainval).astype(np.float32)
X_test_trans = preprocess.transform(X_test).astype(np.float32)

y_trainval_np = y_trainval.to_numpy()
y_test_np = y_test.to_numpy()

final_model = TabNetClassifier(
    seed=5831,
    n_d=best_n_d,
    n_a=best_n_d,
    n_steps=best_n_steps,
    gamma=1.5,
    lambda_sparse=1e-4,
    optimizer_params=dict(lr=best_lr),
    mask_type="entmax",
    verbose=0
)

final_model.fit(
    X_trainval_trans,
    y_trainval_np,
    max_epochs=200,
    batch_size=256,
    virtual_batch_size=64,
    num_workers=0,
    drop_last=False
)

/opt/anaconda3/envs/CS5831FinalClassification/lib/python3.12/site-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)


In [43]:
y_test_prob = final_model.predict_proba(X_test_trans)[:, 1]
y_test_pred = (y_test_prob >= best_threshold).astype(int)

recall_tabnet = recall_score(y_test, y_test_pred, pos_label=1)
precision_tabnet = precision_score(y_test, y_test_pred, pos_label=1)
cm = confusion_matrix(y_test, y_test_pred)

cm_tabnet = pd.DataFrame(
        cm,
        index=["Actual No Allergen", "Actual Has Allergens"],
        columns=["Pred No Allergen", "Pred Has Allergens"]
)
print("Test Recall:", recall_tabnet)
print("Test Precision:", precision_tabnet)
print(cm_tabnet)

Test Recall: 0.8762057877813505
Test Precision: 0.788712011577424
                      Pred No Allergen  Pred Has Allergens
Actual No Allergen                 232                 146
Actual Has Allergens                77                 545


In [44]:
results_dict["Tabnet"] = {"recall":recall_tabnet, "precision": precision_tabnet, "threshold": best_threshold}

In [45]:
model_results_df = pd.DataFrame.from_dict(results_dict, orient="index").reset_index()
model_results_df.rename(columns={"index": "model"}, inplace=True)

model_results_df = model_results_df.sort_values(by=["recall", "precision"], ascending = False).reset_index(drop = True)
model_results_df.index += 1
model_results_df

,model,recall,precision,threshold
1,Lightgbm,0.995177,0.706621,0.05
2,XGBoost,0.990354,0.735962,0.10
3,QDA,0.971061,0.701510,0.60
4,LDA,0.956592,0.706651,0.40
5,Reg LDA,0.932476,0.716934,0.50
6,Tabnet,0.876206,0.788712,0.50
